# Credit Card Transaction & Fraud Pattern Analysis
**Goal:** find rule-based signals of fraud and measure them against the labelled `is_fraud` column using precision and recall.

**Data:** *Credit Card Transactions Fraud Detection Dataset* (Kaggle, kartik2112): simulated transactions. This notebook uses a sample of 500 cards with all of their transactions (662,187 rows), already cleaned (`data/flagged.csv`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("data/flagged.csv", parse_dates=["trans_time"])
print(df.shape)
df[["trans_time", "cc_num", "category", "amt", "is_fraud", "rule_amount", "rule_velocity", "rule_night", "flag_count"]].head()

## 1. Headline numbers
**Why:** fraud is rare, so we first need the base rate. It decides which metrics are meaningful: with so little fraud, plain accuracy would be useless, because a rule that flags nothing would still be right almost every time.

In [ ]:
total = len(df)
fraud = int(df["is_fraud"].sum())
print("Transactions:", total)
print("Fraud transactions:", fraud, f"({fraud / total * 100:.2f}%)")
print("Average amount, normal:", round(df.loc[df.is_fraud == 0, "amt"].mean(), 2))
print("Average amount, fraud :", round(df.loc[df.is_fraud == 1, "amt"].mean(), 2))

## 2. Fraud rate by merchant category
**Why:** shows where fraud concentrates, so analysts know which merchant types deserve closer review.

In [ ]:
cat = df.groupby("category")["is_fraud"].agg(["count", "sum", "mean"])
cat["fraud_rate_%"] = (cat["mean"] * 100).round(2)
cat = cat.sort_values("fraud_rate_%", ascending=False)
print(cat[["count", "sum", "fraud_rate_%"]])
cat["fraud_rate_%"].sort_values().plot(kind="barh", title="Fraud rate by category (%)", figsize=(8, 5))
plt.xlabel("Fraud rate (%)"); plt.ylabel("Category"); plt.show()

## 3. Fraud by hour of day (the problem area)
**Why:** if fraud clusters at certain times, a time-of-day rule can catch it cheaply.

In [ ]:
hour = df.groupby("trans_hour")["is_fraud"].agg(["count", "sum", "mean"])
hour["fraud_rate_%"] = (hour["mean"] * 100).round(2)
night = [22, 23, 0, 1, 2, 3]
print("Share of all fraud between 22:00 and 03:59:",
      round(hour.loc[night, "sum"].sum() / fraud * 100, 1), "%")
colors = ["#c53030" if h in night else "#a0aec0" for h in hour.index]
plt.figure(figsize=(10, 4))
plt.bar(hour.index, hour["fraud_rate_%"], color=colors)
plt.title("Fraud rate by hour of day (night in red)")
plt.xlabel("Hour of day"); plt.ylabel("Fraud rate (%)"); plt.xticks(range(24)); plt.show()

## 4. Fraud by state and by month
**Why:** checks whether fraud is geographic or changes over time. Both views help decide where monitoring effort should go.

In [ ]:
st = df.groupby("state")["is_fraud"].agg(["count", "mean"])
st = st[st["count"] >= 1000].sort_values("mean", ascending=False).head(10)
print((st["mean"] * 100).round(2))
monthly = df.groupby(df["trans_time"].dt.to_period("M").astype(str))["is_fraud"].sum()
monthly.plot(marker="o", figsize=(10, 4), title="Fraud transactions per month")
plt.xlabel("Month"); plt.ylabel("Fraud transactions"); plt.xticks(rotation=60); plt.show()

## 5. Evaluating the three rules
**Why:** a rule is only useful if it flags real fraud (**precision**) without missing too much of it (**recall**).
- Rule 1: amount > 600
- Rule 2: 3 or more transactions on the same card within 30 minutes (velocity)
- Rule 3: night hours (22:00-03:59) and amount > 300

In [ ]:
def evaluate(name, flagged):
    n = int(flagged.sum())
    caught = int((flagged & (df.is_fraud == 1)).sum())
    p = caught / n * 100 if n else 0
    r = caught / fraud * 100
    f1 = 2 * p * r / (p + r) if p + r else 0
    return {"rule": name, "flagged": n, "caught": caught,
            "precision_%": round(p, 1), "recall_%": round(r, 1), "F1": round(f1, 1)}

rows = [
    evaluate("amount > 600", df.rule_amount == 1),
    evaluate("velocity (3 in 30 min)", df.rule_velocity == 1),
    evaluate("night and amt > 300", df.rule_night == 1),
    evaluate("any 1 or more rules", df.flag_count >= 1),
    evaluate("2 or more rules (final)", df.flag_count >= 2),
    evaluate("all 3 rules", df.flag_count == 3),
]
print(pd.DataFrame(rows).to_string(index=False))

## 6. Risk tiers
**Why:** the number of rules that fire works as a confidence score. Analysts can review High first and treat Low as a watch list.

In [ ]:
tiers = df.assign(tier=df.flag_count.map({0: "Clear", 1: "Low", 2: "Medium", 3: "High"})) \
          .groupby("tier")["is_fraud"].agg(["count", "sum", "mean"])
tiers["fraud_rate_%"] = (tiers["mean"] * 100).round(1)
print(tiers.loc[["Clear", "Low", "Medium", "High"], ["count", "sum", "fraud_rate_%"]])

## 7. Insights and recommendations
**Insights (from this sample of 662,187 transactions, 3,771 of them fraud, a 0.57% fraud rate):**
1. **Fraud is concentrated at night.** 85.1% of all fraud happens between 22:00 and 03:59, where the fraud rate is 1.4% to 2.9% per hour, against about 0.1% for the rest of the day.
2. **Online categories are riskiest.** `shopping_net` has a fraud rate of 1.80% and `misc_net` 1.44%, while `home` is only 0.13%.
3. **Fraud amounts sit in a band.** Fraud averages 540.49 versus 66.64 for normal transactions, and fraud rarely exceeds 1,376, so very large purchases are mostly legitimate.
4. **No single rule is enough.** The amount rule has 33.1% precision, velocity only 5.8%, and the night-and-amount rule 43.3%.
5. **Combining rules works.** Requiring 2 or more rules gives 64.1% precision and 42.0% recall (F1 50.7), and the High tier (all 3 rules) reaches 96.9% precision on 161 transactions.
6. **Velocity works as a confirming signal.** On its own it is weak, but together with the night rule it reaches 91.0% precision.

**Recommendations:**
- Send **High** tier transactions (all 3 rules) for automatic hold or immediate review. Precision is 96.9%.
- Put **Medium** tier (2 rules, about 62% precision) in the analyst review queue.
- Add extra checks for online shopping and miscellaneous online categories between 22:00 and 04:00.
- Do not use velocity alone, because 5.8% precision would flood analysts with false alarms.
- Recall is 42.0%, so more than half of fraud is still missed. The next step is to add features such as distance between customer and merchant, or a trained model.

**Limitations:** the data is simulated, and the thresholds were chosen on the same data they are evaluated on, so real-world performance would likely be lower. They should be validated on a separate hold-out file (for example `fraudTest.csv`).